In [1]:
# Cell 1: imports and solver config
import os
import glob
import pandas as pd
import numpy as np
import cobra as cb
import cobra
import scipy.stats as st
import matplotlib.pyplot as plt
import gifba

cobra.Configuration().solver = "glpk"  # no Gurobi needed for giFBA itself

In [2]:
# Cell 2: paths — update these to match your local setup
INVITRO_ORIGINAL = "/Users/jalacy/PhD/GIFBA/invitro_original.csv"   # raw_data/invitro_original.csv from scfa_predictions repo
DM38_MEDIA = "/Users/jalacy/PhD/GIFBA/DM38.csv"                     # media/DM38.csv from scfa_predictions repo
AGORA_DIR = "/Users/jalacy/PhD/UC_project/COBRA/AGORA2_models/AGORA2_all_Mat_files"                  # your local AGORA2 folder

In [3]:
# Cell 3: build low-richness relative abundance table + measured butyrate (panel b)
raw = pd.read_csv(INVITRO_ORIGINAL, index_col=1)

raw[raw.columns[11:37]] = raw[raw.columns[11:37]].fillna(0).astype(int)
raw["richness"] = raw[raw.columns[11:37]].sum(axis=1)
raw["Plate"] = raw["Plate"].astype(str).str.split(".").str[0].str.zfill(2)
raw["Column"] = raw["Column"].astype(str).str.split(".").str[0].str.zfill(2)
raw["Run"] = raw["Sequencing Run"].str[-3:]
raw["sample_id"] = "P" + raw["Plate"] + raw["Row"] + raw["Column"] + "_" + raw["Run"]
raw = raw[raw["Contamination?"] == "No"].set_index("sample_id")

frac_cols = [c for c in raw.columns if "Fraction" in c and c != "B.cereus Fraction"]
rel_abundance = raw[frac_cols].round(4).dropna(how="all")
rel_abundance.columns = [c.split(" ")[0] for c in rel_abundance.columns]  # two-letter codes only

assert rel_abundance.columns[rel_abundance.columns.duplicated()].empty, "duplicate columns found"

low = raw[raw["richness"] <= 5]
rel_abundance_low = rel_abundance.loc[rel_abundance.index.intersection(low.index)]

measured = raw["Butyrate"] / raw["OD"]
measured = measured[(measured >= 0) & (measured <= 100)]
measured_low = measured.loc[measured.index.intersection(rel_abundance_low.index)]

print(rel_abundance_low.shape, "samples x codes")
rel_abundance_low.head()

(895, 30) samples x codes


,BA,CA,BT,BU,PC,AC,BH,CC,CG,ER,...,CS,PJ,FP,EH,EC,BC,HB,BO,DL,BL
sample_id,,,,,,,,,,,,,,,,,,,,,
P19A01_003,0.0,0.0018,0.0,0.0,0.0034,0.0003,0.0,0.0000,0.0,0.9925,...,0.0,0.0,0.0013,0.0,0.0,0.0,0.0,0.0,0.0000,0.0
P19C01_003,0.0,0.0000,0.0,0.0,0.0000,0.7357,0.0,0.0000,0.0,0.2632,...,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0003,0.0
P19E01_003,0.0,0.0000,0.0,0.0,0.0000,0.0000,0.0,0.8989,0.0,0.1006,...,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0002,0.0
P19E01_003,0.0,0.0000,0.0,0.0,0.0000,0.0000,0.0,0.8989,0.0,0.1006,...,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0002,0.0
P19G01_003,0.0,0.0000,0.0,0.0,0.0000,0.0002,0.0,0.0002,0.0,0.5427,...,0.0,0.0,0.0002,0.0,0.0,0.0,0.0,0.0,0.0000,0.0


In [4]:
# Cell 4: build the giFBA media dict from DM38.csv (convert _m suffix to (e) suffix)
media_df = pd.read_csv(DM38_MEDIA, index_col=0)

media = {
    row["reaction"].replace("_m", "(e)"): -abs(row["flux"])
    for _, row in media_df.iterrows()
}

list(media.items())[:5]

[('EX_4abz(e)', -0.0073),
 ('EX_ade(e)', -6e-06),
 ('EX_ala_L(e)', -0.53),
 ('EX_arg_L(e)', -2.2),
 ('EX_asn_L(e)', -0.26)]

In [5]:
# Cell 5: resolve one AGORA2 model file per two-letter code (species match, genus fallback)
code_to_genus = {
    'PC':'Prevotella','PJ':'Parabacteroides','BV':'Bacteroides','BF':'Bacteroides',
    'BO':'Bacteroides','BT':'Bacteroides','BC':'Bacteroides','BY':'Bacteroides',
    'BU':'Bacteroides','DP':'Desulfovibrio','BL':'Bifidobacterium','BA':'Bifidobacterium',
    'BP':'Bifidobacterium','CA':'Collinsella','EL':'Eggerthella','FP':'Faecalibacterium',
    'CH':'Clostridium','AC':'Anaerostipes','BH':'Blautia','CG':'Clostridium',
    'ER':'Eubacterium','RI':'Roseburia','CC':'Coprococcus','DL':'Dorea','DF':'Dorea',
}

code_to_species_guess = {
    "AC": "Anaerostipes_caccae", "DP": "Desulfovibrio_piger", "CG": "Clostridium_asparagiforme",
    "EL": "Eggerthella_lenta", "DF": "Dorea_formicigenerans", "CA": "Collinsella_aerofaciens",
    "RI": "Roseburia_intestinalis", "CH": "Clostridium_hiranonis", "ER": "Eubacterium_rectale",
    "CC": "Coprococcus_comes", "FP": "Faecalibacterium_prausnitzii", "PC": "Prevotella_copri",
    "BV": "Bacteroides_vulgatus", "BO": "Bacteroides_ovatus", "BT": "Bacteroides_thetaiotaomicron",
    "BU": "Bacteroides_uniformis", "BH": "Blautia_hydrogenotrophica", "DL": "Dorea_longicatena",
    "PJ": "Parabacteroides_johnsonii", "BF": "Bacteroides_fragilis", "BC": "Bacteroides_caccae",
    "BL": "Bifidobacterium_longum", "BA": "Bifidobacterium_adolescentis",
    "BP": "Bifidobacterium_pseudocatenulatum", "BY": None,
}

def pick_best(files):
    preferred = [f for f in files if ("DSM" in f or "ATCC" in f)]
    return preferred[0] if preferred else (files[0] if files else None)

code_to_model = {}
for code, genus in code_to_genus.items():
    species = code_to_species_guess.get(code)
    files = glob.glob(os.path.join(AGORA_DIR, f"{species}*.mat")) if species else []
    source = "species"
    if not files:
        files = glob.glob(os.path.join(AGORA_DIR, f"{genus}_*.mat"))
        source = "genus fallback"
    chosen = pick_best(files)
    code_to_model[code] = chosen
    print(f"{code}: {os.path.basename(chosen) if chosen else 'NOT FOUND'}  ({source}, {len(files)} candidates)")

missing = [c for c, f in code_to_model.items() if f is None]
print("\nMissing entirely:", missing)

PC: Prevotella_copri_CB7_DSM_18205.mat  (species, 2 candidates)
PJ: Parabacteroides_johnsonii_DSM_18315.mat  (species, 2 candidates)
BV: Bacteroides_vulgatus_ATCC_8482.mat  (species, 25 candidates)
BF: Bacteroides_fragilis_ERR2221268.mat  (species, 25 candidates)
BO: Bacteroides_ovatus_ATCC_8483.mat  (species, 10 candidates)
BT: Bacteroides_thetaiotaomicron_ERR2221274.mat  (species, 12 candidates)
BC: Bacteroides_caccae_ATCC_43185.mat  (species, 6 candidates)
BY: Bacteroides_pyogenes_DSM20611.mat  (genus fallback, 216 candidates)
BU: Bacteroides_uniformis_ATCC_8492.mat  (species, 24 candidates)
DP: Desulfovibrio_piger_ATCC_29098.mat  (species, 1 candidates)
BL: Bifidobacterium_longum_infantis_ATCC_15697.mat  (species, 28 candidates)
BA: Bifidobacterium_adolescentis_ATCC_15703.mat  (species, 13 candidates)
BP: Bifidobacterium_pseudocatenulatum_DSM_20438.mat  (species, 6 candidates)
CA: Collinsella_aerofaciens_ATCC_25986.mat  (species, 18 candidates)
EL: Eggerthella_lenta_DSM_11863.mat  

In [ ]:
# Cell 6: run giFBA per low-richness sample, collect predicted butyrate (fixed filtering)
BUTYRATE_EX = "EX_but(e)"

_model_cache = {}
def load_code_model(code):
    if code not in _model_cache:
        _model_cache[code] = cb.io.load_matlab_model(code_to_model[code])
    return _model_cache[code]

predicted_butyrate = {}
skipped_unknown_codes = set()

for sample_id, row in rel_abundance_low.iterrows():
    present = row[row > 0]

    unknown = present.index.difference(code_to_model.keys())
    if len(unknown) > 0:
        skipped_unknown_codes.update(unknown)
    present = present[present.index.isin(code_to_model.keys())]

    present = present[[code_to_model[c] is not None for c in present.index]]
    if present.empty:
        continue

    codes_present = present.index.tolist()
    abund = (present / present.sum()).tolist()
    models = [load_code_model(c) for c in codes_present]

    community = gifba.gifbaObject(models, media, rel_abund=abund)
    media_flux, org_flux = community.run_gifba(iters=5, method="pfba", v=False)

    predicted_butyrate[sample_id] = org_flux[BUTYRATE_EX].sum() if BUTYRATE_EX in org_flux.columns else None

predicted_butyrate = pd.Series(predicted_butyrate, name="predicted")
print(f"{predicted_butyrate.notna().sum()} / {len(rel_abundance_low)} samples produced a prediction")
print("Unknown codes encountered (dropped):", skipped_unknown_codes)
predicted_butyrate.head()

No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e, p
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expressi

Relative abundances set to: [1.80054016e-03 3.40102031e-03 3.00090027e-04 9.92797839e-01
 2.00060018e-04 2.00060018e-04 1.30039012e-03]


No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e


Relative abundances set to: [7.35994398e-01 2.63305322e-01 2.00080032e-04 2.00080032e-04
 3.00120048e-04]


No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e, p
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e, p
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expre

...

Relative abundances set to: [4.00160064e-04 3.20128051e-03 4.00160064e-04 4.60184074e-03
 4.00160064e-04 2.00080032e-04 2.00080032e-04 2.00080032e-04
 2.00080032e-04 9.90196078e-01]


No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e, p
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e


Relative abundances set to: [1.99980002e-04 1.99980002e-04 1.99980002e-04 1.99980002e-04
 1.99980002e-04 6.51934807e-01 3.41365863e-01 1.99980002e-04
 1.99980002e-04 3.99960004e-03 1.29987001e-03]


No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e, p
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e, p
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e, p
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e


Relative abundances set to: [1.99960008e-04 2.57548490e-01 7.40851830e-01 1.99960008e-04
 9.99800040e-04 1.99960008e-04]
Relative abundances set to: [9.90094056e-01 2.00120072e-04 4.00240144e-04 2.00120072e-04
 4.00240144e-04 7.90474285e-03 2.00120072e-04 2.00120072e-04
 2.00120072e-04 2.00120072e-04]
Relative abundances set to: [1.00020004e-04 3.00060012e-04 6.00120024e-04 3.00060012e-04
 1.00020004e-04 6.00120024e-04 9.18683737e-01 1.00020004e-04
 7.92158432e-02]
Relative abundances set to: [2.00040008e-04 9.86997399e-01 2.00040008e-04 4.00080016e-04
 2.00040008e-04 1.14022805e-02 2.00040008e-04 4.00080016e-04]
Relative abundances set to: [9.90096038e-01 1.00040016e-04 3.60144058e-03 5.00200080e-04
 1.00040016e-04 2.00080032e-04 5.20208083e-03 2.00080032e-04]
Relative abundances set to: [1.99960008e-04 3.94821036e-01 1.99960008e-04 6.04179164e-01
 1.99960008e-04 1.99960008e-04 1.99960008e-04]
Relative abundances set to: [4.00040004e-04 2.00020002e-04 9.80198020e-01 1.86018602e-02
 2.